In [12]:
import pandas as pd

In [13]:
from sklearn.datasets import fetch_covtype
import pandas as pd
import numpy as np

data = fetch_covtype()
X = data.data
y = data.target - 1

# Just for observation, not used anywhere else
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
print(df.head())
print(df.describe())
print(df['target'].value_counts())
del df  # deleting it after, free the memory

   Elevation  Aspect  Slope  Horizontal_Distance_To_Hydrology  \
0     2596.0    51.0    3.0                             258.0   
1     2590.0    56.0    2.0                             212.0   
2     2804.0   139.0    9.0                             268.0   
3     2785.0   155.0   18.0                             242.0   
4     2595.0    45.0    2.0                             153.0   

   Vertical_Distance_To_Hydrology  Horizontal_Distance_To_Roadways  \
0                             0.0                            510.0   
1                            -6.0                            390.0   
2                            65.0                           3180.0   
3                           118.0                           3090.0   
4                            -1.0                            391.0   

   Hillshade_9am  Hillshade_Noon  Hillshade_3pm  \
0          221.0           232.0          148.0   
1          220.0           235.0          151.0   
2          234.0           238.0   

In [14]:
from sklearn.model_selection import train_test_split
import numpy as np
print(np.isnan(X).sum())
print(np.isnan(y).sum())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

0
0


In [15]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import time as tm

xgb = XGBClassifier()
lgbm = LGBMClassifier()

xgb_start_time = tm.time()
xgb.fit(X_train, y_train)
xgb_end_time = tm.time()
xgb_time_to_fit = xgb_end_time - xgb_start_time


lgbm_start_time = tm.time()
lgbm.fit(X_train, y_train)
lgbm_end_time = tm.time()
lgbm_time_to_fit = lgbm_end_time - lgbm_start_time

print(f'It took xgb {xgb_time_to_fit} seconds to fit')
print(f'It took lgbm {lgbm_time_to_fit} secpmds to fit')


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009371 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2263
[LightGBM] [Info] Number of data points in the train set: 464809, number of used features: 53
[LightGBM] [Info] Start training from score -1.008939
[LightGBM] [Info] Start training from score -0.718264
[LightGBM] [Info] Start training from score -2.788115
[LightGBM] [Info] Start training from score -5.354079
[LightGBM] [Info] Start training from score -4.114268
[LightGBM] [Info] Start training from score -3.510169
[LightGBM] [Info] Start training from score -3.343858
It took xgb 12.622357606887817 seconds to fit
It took lgbm 4.7380945682525635 secpmds to fit


In [16]:
xgb_train = xgb.score(X_train, y_train)
lgbm_train = lgbm.score(X_train, y_train)

xgb_test = xgb.score(X_test, y_test)
lgbm_test = lgbm.score(X_test, y_test)

print(f'The xgb train score is {xgb_train}')
print(f'The lgbm train score is {lgbm_train}')

print(f'The xgb test score is {xgb_test}')
print(f'The lgbm test score is {lgbm_test}')

C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


The xgb train score is 0.8799270237882657
The lgbm train score is 0.8647831690006003
The xgb test score is 0.868936258100049
The lgbm test score is 0.8556061375351756


In [17]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
import numpy as np
lgbm_params = {
    'n_estimators': [300, 500, 1000], 
    'num_leaves': np.arange(20, 150, 10),
    'max_depth': np.arange(-1, 10, 1), 
    'learning_rate': [0.01, 0.05, 0.1], 
    'subsample': np.arange(0.5, 1.0, 0.1), 
    'colsample_bytree': np.arange(0.5, 1.0, 0.1)
}

xgb_params = {
    'n_estimators': [300, 500, 1000], 
    'max_depth': np.arange(3, 10, 1), 
    'learning_rate': [0.01, 0.05, 0.1], 
    'subsample': np.arange(0.5, 1.0, 0.1), 
    'colsample_bytree': np.arange(0.5, 1.0, 0.1)
}

lgbm_search = HalvingRandomSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_params, 
    cv=5,
    n_jobs=-1,
    random_state=0,
    min_resources=10000, 
    scoring='f1_macro', 
    verbose=1
)

xgb_search = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=xgb_params, 
    cv=5,
    n_jobs=-1,
    random_state=0,
    min_resources=10000, 
    scoring='f1_macro', 
    verbose=1
)

In [18]:
# Recording the amount of time it takes to fit both searches 
import time
import warnings
warnings.filterwarnings('ignore')

lgbm_start_time = time.time()
lgbm_search.fit(X_train, y_train)
lgbm_end_time = time.time()
lgbm_time_to_fit = lgbm_end_time - lgbm_start_time

n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 4
min_resources_: 10000
max_resources_: 464809
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 46
n_resources: 10000
Fitting 5 folds for each of 46 candidates, totalling 230 fits
----------
iter: 1
n_candidates: 16
n_resources: 30000
Fitting 5 folds for each of 16 candidates, totalling 80 fits
----------
iter: 2
n_candidates: 6
n_resources: 90000
Fitting 5 folds for each of 6 candidates, totalling 30 fits
----------
iter: 3
n_candidates: 2
n_resources: 270000
Fitting 5 folds for each of 2 candidates, totalling 10 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008967 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2263
[LightGBM] [Info] Number of data points in the train set: 464809, number of used features: 53
[LightGBM] [Info] Start training from score -1.008939
[LightGBM] [Info] Start training from score

KeyboardInterrupt: 

In [24]:
xgb_search = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=xgb_params,
    cv=3,
    n_jobs=-1,
    random_state=0,
    min_resources=10000,
    factor=4,
    scoring='f1_macro',
    verbose=1
)

xgb_start_time = time.time()
xgb_search.fit(X_train, y_train)
xgb_end_time = time.time()
xgb_time_to_fit = xgb_end_time - xgb_start_time
print(f'XGBoost search took {xgb_time_to_fit:.2f} seconds')

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 3
min_resources_: 10000
max_resources_: 464809
aggressive_elimination: False
factor: 4
----------
iter: 0
n_candidates: 46
n_resources: 10000
Fitting 3 folds for each of 46 candidates, totalling 138 fits
----------
iter: 1
n_candidates: 12
n_resources: 40000
Fitting 3 folds for each of 12 candidates, totalling 36 fits
----------
iter: 2
n_candidates: 3
n_resources: 160000
Fitting 3 folds for each of 3 candidates, totalling 9 fits
XGBoost search took 884.22 seconds


In [25]:
print(lgbm_search.best_params_)
print(lgbm_search.best_score_)

print(xgb_search.best_params_)
print(xgb_search.best_score_)

{'subsample': np.float64(0.6), 'num_leaves': np.int64(100), 'n_estimators': 1000, 'max_depth': np.int64(-1), 'learning_rate': 0.05, 'colsample_bytree': np.float64(0.7)}
0.9256371947602993
{'subsample': np.float64(0.7), 'n_estimators': 1000, 'max_depth': np.int64(9), 'learning_rate': 0.1, 'colsample_bytree': np.float64(0.7)}
0.8972235917449402


In [28]:
print(f'XGBoost search took {xgb_time_to_fit:.2f} seconds')
print(f'LGBM search took {lgbm_time_to_fit:.2f} seconds')


XGBoost search took 884.22 seconds
LGBM search took 4190.60 seconds


In [29]:
lgbm_best = lgbm_search.best_estimator_
xgb_best = xgb_search.best_estimator_

lgbm_train_results = lgbm_best.score(X_train, y_train)
xgb_train_results = xgb_best.score(X_train, y_train)

lgbm_test_results = lgbm_best.score(X_test, y_test)
xgb_test_results = xgb_best.score(X_test, y_test)


print(f'lgbm train score: {lgbm_train_results}')
print(f'lgbm test score: {lgbm_test_results}')

print(f'\nxgbtrain score: {xgb_train_results}')
print(f'xgb test score: {xgb_test_results}')


lgbm train score: 0.9880725201104109
lgbm test score: 0.9615500460401194

xgbtrain score: 0.9979496954663098
xgb test score: 0.9667564520709448


In [30]:
from sklearn.metrics import classification_report
print(classification_report(y_test, lgbm_best.predict(X_test)))
print(classification_report(y_test, xgb_best.predict(X_test)))

              precision    recall  f1-score   support

           0       0.96      0.95      0.96     42368
           1       0.96      0.97      0.97     56661
           2       0.96      0.97      0.97      7151
           3       0.92      0.88      0.90       549
           4       0.94      0.88      0.91      1899
           5       0.95      0.94      0.94      3473
           6       0.98      0.97      0.97      4102

    accuracy                           0.96    116203
   macro avg       0.95      0.94      0.94    116203
weighted avg       0.96      0.96      0.96    116203

              precision    recall  f1-score   support

           0       0.97      0.96      0.97     42368
           1       0.97      0.98      0.97     56661
           2       0.96      0.97      0.96      7151
           3       0.90      0.87      0.89       549
           4       0.95      0.88      0.91      1899
           5       0.94      0.93      0.94      3473
           6       0.97 

# Day 12 | Boosting Bake-Off Results

## Dataset
- **Covertype** — 581,012 samples, 54 features, 7 classes
- Notable class imbalance: Class 4 had only 2,747 samples vs 283,301 for Class 2

## Benchmark Summary

| | XGBoost | LightGBM |
|---|---|---|
| Default Accuracy | 0.8689 | 0.8556 |
| Tuned Accuracy | 0.9668 | 0.9616 |
| Macro F1 | 0.94 | 0.94 |
| Search Time | ~15 min | ~70 min |
| Overfitting Gap | 0.031 | 0.026 |

## Key Findings

- XGBoost wins marginally on accuracy (+0.005)
- LightGBM overfits less and is more memory efficient
- Both models struggled equally on minority classes (Class 3, Class 4)
- Macro F1 is identical — on balanced evaluation, neither model is superior
- Search time difference was largely due to cv=5 vs cv=3, not model complexity

## Production Verdict

- **Choose XGBoost** when accuracy is the priority and retraining is infrequent
- **Choose LightGBM** when data is massive, retraining is frequent, or memory is constrained
- With proper tuning, the accuracy gap shrinks to negligible on most real-world tasks

In [31]:
import joblib

joblib.dump(lgbm_best, 'lgbm-best.pkl')
joblib.dump(xgb_best, 'xgb-best.pkl')

['xgb-best.pkl']